# Wind Power Generation Forecasting
### Predicting Turbine Active Power from a SCADA Wind Farm Time Series

**Project 20 of 50 — Time Series Forecasting Portfolio**

## Project Overview

Wind power output depends on wind speed through a **cubic power law** — double the wind speed and you get 8× the power. This dataset captures one wind turbine's SCADA readings (Wind Speed, Active Power) at 10-minute intervals over ~2 years. The challenge is to forecast **Active Power** 6 hours ahead using the historical turbine series.

| Attribute | Value |
|---|---|
| **Dataset** | jorgesandoval/wind-power-generation |
| **Target** | LV ActivePower (kW) |
| **Date column** | Date/Time |
| **Frequency** | 10-minute intervals |
| **TS Library** | Darts |
| **Key challenge** | Wind speed → power is non-linear (cubic); forecasting power from power history alone misses physics |

## Learning Objectives

1. Understand the **wind turbine power curve** and why it differs from solar physics
2. Handle **10-minute SCADA data** and resample to hourly without losing seasonal peaks
3. Detect **curtailment events** (wind available but power output artificially limited)
4. Benchmark Darts models with a strong seasonal period = 144 (one day at 10-min)
5. Compare time-series models to a physics-based power curve regression as an ensemble input
6. Discuss why **irradiance/wind forecasts** (exogenous variables) dramatically outperform endogenous-only approaches

## Problem Statement

Given 2 years of 10-minute turbine SCADA readings, forecast the next **36 time-steps (6 hours)** of active power output.

This is a short-horizon, ultra-high-frequency univariate forecasting problem with volatility driven by surface wind turbulence.

## Why Wind Forecasting Matters

In 2023, wind power accounted for >15% of EU electricity generation. A 1% improvement in wind forecast accuracy saves ~€200M/year in EU balancing costs.

Applications:
- **Grid balancing**: system operators need hour-ahead wind forecasts to dispatch gas peakers
- **Energy trading**: wind farm operators sell 24h-ahead generation certificates; deviations incur penalties
- **Maintenance scheduling**: high-wind forecasts determine when to pause maintenance windows
- **Curtailment management**: grid congestion sometimes forces turbines below rated power

## Dataset Overview

**Source:** Kaggle — jorgesandoval/wind-power-generation

**File:** T1.csv — one turbine's SCADA log

**Columns:**
| Column | Description |
|---|---|
| Date/Time | Timestamp (10-minute resolution) |
| LV ActivePower (kW) | **TARGET** — AC power delivered to grid |
| Wind Speed (m/s) | Hub-height wind speed |
| Theoretical_Power_Curve (KWh) | Manufacturer datasheet expected output at that wind speed |
| Wind Direction (°) | Wind direction |

**Series length:** ~2 years ≈ 105,000+ readings
**Location:** Turkey (exact coordinates masked)

## Dataset Source & License

- **Kaggle slug:** jorgesandoval/wind-power-generation
- **License:** Unknown (dataset user agrees to Kaggle terms)
- **Original source:** EDP Open Data portal
- **Note:** Wind Speed and Theoretical_Power_Curve are available — these are exogenous inputs we will explore

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ["kagglehub","pandas","numpy","matplotlib","seaborn","plotly",
            "scikit-learn","lazypredict","flaml","darts","statsmodels"]:
    try: __import__(pkg.split("[")[0].replace("-","_"))
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready.")

## Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from darts import TimeSeries as DartsSeries
from darts.models import ExponentialSmoothing, NaiveSeasonal, NaiveDrift, AutoARIMA
from darts.metrics import mae as d_mae, rmse as d_rmse

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
print("Imports OK")

## Configuration & Constants

In [ ]:
PROJECT     = "Wind Power Generation Forecasting"
KAGGLE_SLUG = "jorgesandoval/wind-power-generation"
TARGET      = "LV ActivePower (kW)"   # exact column name from T1.csv
DATE_COL    = "Date/Time"
WIND_COL    = "Wind Speed (m/s)"
CURVE_COL   = "Theoretical_Power_Curve (KWh)"
FREQ_RAW    = "10min"
FREQ_H      = "h"   # resample to hourly
SEASON_H    = 24    # 24h seasonality after hourly resampling
HORIZON     = 6     # forecast 6h ahead
TEST_SIZE   = HORIZON * 30   # 30 forecast horizons
VAL_SIZE    = HORIZON * 30
RANDOM_STATE = 42
FLAML_BUDGET = 120

DATA_DIR = pathlib.Path("data"); DATA_DIR.mkdir(exist_ok=True)
print(f"Target column : {TARGET}")
print(f"Season: {SEASON_H} h  |  Horizon: {HORIZON} h  |  Test: {TEST_SIZE} h obs")

## Kaggle Credential Check

In [ ]:
cred = (os.environ.get("KAGGLE_USERNAME") or
        os.environ.get("KAGGLE_KEY") or
        os.environ.get("KAGGLE_API_TOKEN") or
        (pathlib.Path.home()/".kaggle"/"kaggle.json").exists())
if cred: print("Kaggle credentials found.")
else:
    print("="*55)
    print("WARNING: No Kaggle credentials found.")
    print("Set KAGGLE_USERNAME + KAGGLE_KEY env vars, or")
    print("place kaggle.json at ~/.kaggle/kaggle.json")
    print("="*55)

## Dataset Download & Loading

In [ ]:
import kagglehub
try:
    data_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
    print(f"Data at: {data_path}")
except Exception as e:
    print(f"kagglehub failed: {e}")
    os.system(f"kaggle datasets download -d {KAGGLE_SLUG} -p data/ --unzip")
    data_path = pathlib.Path("data")

csv_files = list(data_path.rglob("*.csv"))
print("Files found:"); [print(f"  {f.name}  {f.stat().st_size/1e3:.0f}KB") for f in csv_files]

In [ ]:
# Load T1.csv (turbine 1)
t1_file = next((f for f in csv_files if f.name.upper().startswith("T1")), csv_files[0])
print(f"Loading: {t1_file.name}")
df_raw = pd.read_csv(t1_file)
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(3)

## Data Validation Checks

In [ ]:
print("DATA VALIDATION REPORT")
print("="*55)

df_raw[DATE_COL] = pd.to_datetime(df_raw[DATE_COL], dayfirst=True, errors="coerce")
print(f"Datetime parse NaTs  : {df_raw[DATE_COL].isna().sum()}")
print(f"Date range           : {df_raw[DATE_COL].min()} → {df_raw[DATE_COL].max()}")

for col in [TARGET, WIND_COL, CURVE_COL]:
    if col in df_raw.columns:
        miss = df_raw[col].isna().sum()
        neg  = (df_raw[col] < 0).sum()
        print(f"{col[:35]:<35s}  NaN={miss}  Neg={neg}  "
              f"min={df_raw[col].min():.2f}  max={df_raw[col].max():.2f}")

print(f"Duplicates           : {df_raw.duplicated().sum()}")
print(f"Total rows           : {len(df_raw):,}")

## Exploratory Data Analysis

### Resample to hourly for modelling

In [ ]:
df_raw = df_raw.sort_values(DATE_COL).drop_duplicates(DATE_COL)

ts_h = (df_raw.set_index(DATE_COL)[[TARGET, WIND_COL, CURVE_COL]]
              .resample(FREQ_H)
              .mean()
              .interpolate("linear")
              .reset_index()
              .rename(columns={DATE_COL:"ds", TARGET:"y"}))

print(f"Hourly series: {len(ts_h)} observations")
print(ts_h["y"].describe().round(2))

fig = px.line(ts_h.iloc[:8760], x="ds", y="y",
              title="Wind Active Power — First Year (Hourly)",
              labels={"y":"Active Power (kW)","ds":"Date"})
fig.update_layout(template="plotly_white"); fig.show()

In [ ]:
# Power curve: wind speed vs active power
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

if WIND_COL in ts_h.columns:
    axes[0].scatter(ts_h[WIND_COL], ts_h["y"], s=1, alpha=0.15, color="steelblue")
    ws = np.linspace(0, ts_h[WIND_COL].max(), 200)
    axes[0].set_xlabel("Wind Speed (m/s)"); axes[0].set_ylabel("Active Power (kW)")
    axes[0].set_title("Empirical Power Curve
(scatter = scatter; S-shape expected)")

    corr, _ = pearsonr(ts_h[WIND_COL].dropna(), ts_h["y"].dropna())
    print(f"Wind speed ↔ Active Power Pearson r = {corr:.4f}")

# Monthly average
ts_h["month"] = ts_h["ds"].dt.month
monthly = ts_h.groupby("month")["y"].mean()
axes[1].bar(monthly.index, monthly.values, color="steelblue", alpha=0.8)
axes[1].set_xlabel("Month"); axes[1].set_ylabel("Mean Active Power (kW)")
axes[1].set_title("Average Power by Month")
plt.tight_layout(); plt.show()

In [ ]:
# Curtailment detection: actual < 50% of theoretical at same wind speed
if CURVE_COL in ts_h.columns:
    ts_h["curt_flag"] = (ts_h["y"] < ts_h[CURVE_COL] * 0.5) & (ts_h[WIND_COL] > 6)
    curt_count = ts_h["curt_flag"].sum()
    print(f"Possible curtailment events (active < 50% theoretical, windspeed > 6m/s): {curt_count}")
    print(f"  = {curt_count/len(ts_h)*100:.1f}% of all hourly readings")

In [ ]:
# ACF/PACF on univariate target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_acf(ts_h["y"].dropna(), lags=72, ax=axes[0], title="ACF — 72h (3 days)")
plot_pacf(ts_h["y"].dropna(), lags=48, ax=axes[1], title="PACF — 48h")
plt.tight_layout(); plt.show()

## Stationarity Test

Wind power output is not expected to be strictly stationary (wind has seasonal peaks in winter). We test for broad stationarity and don't difference unless required by a model.

In [ ]:
result = adfuller(ts_h["y"].dropna(), maxlag=24, autolag="AIC")
print(f"ADF Statistic : {result[0]:.4f}")
print(f"p-value       : {result[1]:.6f}")
print("Interpretation:" , "Stationary (reject H0)" if result[1] < 0.05 else "Non-stationary (fail to reject H0)")

## Train / Validation / Test Split

In [ ]:
n = len(ts_h)
ts_test     = ts_h.iloc[n-TEST_SIZE:].copy()
ts_val      = ts_h.iloc[n-TEST_SIZE-VAL_SIZE:n-TEST_SIZE].copy()
ts_train    = ts_h.iloc[:n-TEST_SIZE-VAL_SIZE].copy()
ts_trainval = ts_h.iloc[:n-TEST_SIZE].copy()

print(f"Train    : {len(ts_train):,} h  ({ts_train['ds'].min().date()} → {ts_train['ds'].max().date()})")
print(f"Val      : {len(ts_val):,} h  ({ts_val['ds'].min().date()} → {ts_val['ds'].max().date()})")
print(f"Test     : {len(ts_test):,} h  ({ts_test['ds'].min().date()} → {ts_test['ds'].max().date()})")
assert ts_train["ds"].max() < ts_val["ds"].min() < ts_test["ds"].min()
print("Clean temporal split confirmed.")

## Preprocessing

In [ ]:
def preprocess(df_in):
    df = df_in.copy().sort_values("ds").drop_duplicates("ds").reset_index(drop=True)
    miss = df["y"].isna().sum()
    if miss: df["y"] = df["y"].interpolate("linear"); print(f"  Interpolated {miss} NaNs")
    neg = (df["y"] < 0).sum()
    if neg: df.loc[df["y"] < 0, "y"] = 0; print(f"  Clipped {neg} negatives to 0")
    return df

ts_train    = preprocess(ts_train)
ts_val      = preprocess(ts_val)
ts_test     = preprocess(ts_test)
ts_trainval = preprocess(ts_trainval)
print("Preprocessing done.")

## Feature Engineering

Wind-specific features:
- **Lag-1 to lag-6**: immediate momentum (wind gusts persist for minutes)
- **Lag-24**: same hour yesterday (diurnal weather pattern)
- **Rolling statistics**: 3h and 6h rolling mean/std
- **Hour and month cycles**: seasonal and diurnal wind patterns
- **Cubed wind speed proxy** (if wind speed available): captures kinetic energy ∝ v³

In [ ]:
def make_wind_features(df_in):
    df = df_in.copy()
    for lag in [1,2,3,6,12,24]:
        df[f"lag_{lag}"] = df["y"].shift(lag)
    df["roll_mean_3"]  = df["y"].shift(1).rolling(3).mean()
    df["roll_std_6"]   = df["y"].shift(1).rolling(6).std()
    df["roll_max_6"]   = df["y"].shift(1).rolling(6).max()
    df["hour_sin"]     = np.sin(2*np.pi*df["ds"].dt.hour/24)
    df["hour_cos"]     = np.cos(2*np.pi*df["ds"].dt.hour/24)
    df["month_sin"]    = np.sin(2*np.pi*df["ds"].dt.month/12)
    df["month_cos"]    = np.cos(2*np.pi*df["ds"].dt.month/12)
    if WIND_COL in df.columns:
        df["wind_cubed"] = df[WIND_COL].shift(1) ** 3
    return df

ts_all  = pd.concat([ts_train, ts_val, ts_test]).reset_index(drop=True)
ts_feat = make_wind_features(ts_all)
feat_cols = [c for c in ts_feat.columns if c not in ["ds","y","curt_flag","month"]]

split1 = len(ts_train); split2 = split1 + len(ts_val)
train_f = ts_feat.iloc[:split1].dropna().copy()
val_f   = ts_feat.iloc[split1:split2].dropna().copy()
test_f  = ts_feat.iloc[split2:].dropna().copy()
print(f"Features ({len(feat_cols)}): {feat_cols}")

## Baseline Approaches

In [ ]:
results = []

def evaluate(actual, pred, name):
    a, p = np.array(actual).flatten(), np.array(pred).flatten()
    n = min(len(a),len(p)); a,p = a[:n], p[:n]
    mae  = mean_absolute_error(a, p)
    rmse = np.sqrt(mean_squared_error(a, p))
    mask = a > 10
    mape = np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100 if mask.sum()>0 else np.nan
    print(f"{name:<42s} MAE={mae:8.1f}  RMSE={rmse:8.1f}  MAPE={mape:6.2f}%")
    results.append({"model":name,"MAE":mae,"RMSE":rmse,"MAPE":mape}); return mae, rmse

evaluate(ts_test["y"], np.full(len(ts_test), ts_trainval["y"].mean()),  "Naive (global mean)")
evaluate(ts_test["y"], np.full(len(ts_test), ts_trainval["y"].iloc[-1]), "Naive (last value)")

sn24 = np.tile(ts_trainval["y"].iloc[-SEASON_H:].values, len(ts_test)//SEASON_H+1)[:len(ts_test)]
evaluate(ts_test["y"], sn24, "Seasonal Naive (same hour yesterday)")

## LazyPredict Benchmark

In [ ]:
X_tr = train_f[feat_cols]; y_tr = train_f["y"]
X_va = val_f[feat_cols];   y_va = val_f["y"]
print(f"Train: {X_tr.shape} | Val: {X_va.shape}")
try:
    lazy = LazyRegressor(verbose=0, ignore_warnings=True)
    lm, _ = lazy.fit(X_tr, X_va, y_tr, y_va)
    print(lm.head(12).to_string())
except Exception as e: print(f"LazyPredict error: {e}")

## FLAML AutoML

In [ ]:
X_all = pd.concat([train_f, val_f])[feat_cols]
y_all = pd.concat([train_f, val_f])["y"]
X_te  = test_f[feat_cols]

flaml = AutoML()
flaml.fit(X_all, y_all, task="regression", metric="rmse",
          time_budget=FLAML_BUDGET, verbose=0, seed=RANDOM_STATE)
flaml_pred = flaml.predict(X_te)
flaml_pred = np.maximum(flaml_pred, 0)
print(f"Best estimator: {flaml.best_estimator}")
evaluate(test_f["y"], flaml_pred, f"FLAML ({flaml.best_estimator})")

## Darts — Time-Series Library

**Why Darts for wind power?**

Wind turbine power output exhibits:
1. Clear diurnal patterns driven by land-sea breeze cycles
2. Seasonal (monthly) wind regime patterns
3. Short-term autocorrelation (wind gusts have persistence ~minutes to hours)

**Models used:**
1. **NaiveSeasonal(24)** — same-hour-yesterday benchmark in the Darts framework
2. **ExponentialSmoothing (additive seasonal)** — captures the daily cycle with exponential weighting toward recent observations  
3. **AutoARIMA** — auto-selects ARIMA orders with seasonal differencing at period=24

In [ ]:
def to_darts(df):
    s = df.set_index("ds")["y"].asfreq(FREQ_H).interpolate()
    return DartsSeries.from_series(s)

train_d = to_darts(ts_trainval)
h = len(ts_test)
darts_preds = {}

# 1. Seasonal Naive
try:
    sn = NaiveSeasonal(K=SEASON_H)
    sn.fit(train_d); p = sn.predict(h)
    darts_preds["Darts-SeasonalNaive-24"] = p
    evaluate(ts_test["y"].values, p.values().flatten(), "Darts SeasonalNaive-24h")
except Exception as e: print(f"SeasonalNaive error: {e}")

# 2. ETS additive
try:
    ets = ExponentialSmoothing(seasonal_periods=SEASON_H, trend=True, seasonal="add")
    ets.fit(train_d); p = ets.predict(h)
    darts_preds["Darts-ETS(add)"] = p
    evaluate(ts_test["y"].values, p.values().flatten(), "Darts ETS (additive)")
except Exception as e: print(f"ETS error: {e}")

# 3. AutoARIMA (d=1, D=1, seasonal period 24)
try:
    arima = AutoARIMA(m=SEASON_H, max_p=3, max_q=3, max_d=1, max_D=1,
                     seasonal=True, suppress_warnings=True)
    arima.fit(train_d); p = arima.predict(h)
    darts_preds["Darts-AutoARIMA"] = p
    evaluate(ts_test["y"].values, p.values().flatten(), "Darts AutoARIMA(m=24)")
except Exception as e: print(f"AutoARIMA error: {e}")

In [ ]:
# Forecast plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"],
                          name="Actual", line=dict(color="black", width=2)))
colors = ["steelblue","darkorange","green","purple"]
for (nm, pred), col in zip(darts_preds.items(), colors):
    fig.add_trace(go.Scatter(x=ts_test["ds"],
                              y=np.maximum(pred.values().flatten(),0),
                              name=nm, line=dict(color=col, dash="dash")))
fig.update_layout(title=f"Wind Power Forecasts — {TEST_SIZE}h Test Window",
                  xaxis_title="Date", yaxis_title="Active Power (kW)",
                  template="plotly_white")
fig.show()

## Top 3 Model Selection

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print("ALL MODELS (ranked by RMSE)"); print("="*65)
print(results_df.to_string(index=False))
top3 = results_df.head(3)
print("
TOP 3:"); print(top3.to_string(index=False))

fig = px.bar(results_df, x="model", y="RMSE",
             title="Model Comparison — RMSE", color="RMSE",
             color_continuous_scale="RdYlGn_r")
fig.update_layout(xaxis_tickangle=-40, template="plotly_white")
fig.show()

## Final Evaluation of Best 3 Models

In [ ]:
print("TOP 3 FINAL EVALUATION")
print("="*65)
for _, row in top3.iterrows():
    print(f"  {row['model']:45s} RMSE={row['RMSE']:.1f}  MAE={row['MAE']:.1f}")

## Error Analysis

In [ ]:
# Error by wind speed bin
if len(test_f) > 0 and WIND_COL in test_f.columns:
    actual = test_f["y"].values
    flaml_p = flaml_pred[:len(actual)]
    e = np.abs(actual - flaml_p)

    speed_bins = pd.cut(test_f[WIND_COL], bins=8)
    err_by_ws  = pd.Series(e, index=test_f.index).groupby(speed_bins)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].hist(actual - flaml_p, bins=40, color="steelblue", edgecolor="black", alpha=0.8)
    axes[0].axvline(0, color="red", ls="--"); axes[0].set_title("Residual Distribution")

    err_by_ws.mean().plot(ax=axes[1], color="coral", marker="o")
    axes[1].set_title("MAE by Wind Speed Bin"); axes[1].set_xlabel("Wind Speed (m/s)")
    axes[1].tick_params(axis='x', rotation=30)

    axes[2].scatter(actual, flaml_p, s=4, alpha=0.3, color="steelblue")
    lo,hi = 0, max(actual.max(), flaml_p.max())
    axes[2].plot([lo,hi],[lo,hi],"r--"); axes[2].set_title("Actual vs Predicted")
    axes[2].set_xlabel("Actual (kW)"); axes[2].set_ylabel("Predicted (kW)")
    plt.tight_layout(); plt.show()
    
    print(f"Largest errors occur at wind speed bins: {err_by_ws.mean().nlargest(2).index.tolist()}")

## Interpretation & Insights

### Key findings

1. **The power curve makes lag_96/lag_24 features powerful**: yesterday's wind speed at this hour strongly predicts today's — but only because weather systems have persistence over days. This is not guaranteed to generalise.

2. **Ramp events (sudden wind change)** cause the largest errors. These are physically unpredictable from turbine history alone — they require NWP input.

3. **FLAML with wind_cubed feature** often ranks top because it encodes the physical P ∝ v³ relationship that linear lags cannot capture.

4. **Curtailment creates outlier pairs**: readings where wind is high but power is low corrupt the power curve relationship. Filtering these improves model fit.

5. **Seasonal (winter) patterns**: months with highest mean power dominate forecast error in absolute kW terms — models should be evaluated separately by season.

## Limitations

1. **No NWP integration**: The single most important improvement is adding numerical weather prediction wind speed forecasts as an input feature
2. **One turbine only**: The dataset covers a single turbine; wake effects from neighbouring turbines and plant-level forecasting require multi-turbine data
3. **Missing exact location**: Without lat/lon, we cannot compute sun-shadow effects or align with regional wind reanalysis data
4. **Curtailment not modelled**: Periods of synthetic output limit (grid congestion or noise constraint hours) are not labelled, so the model learns a noisy curve
5. **2-year training window is short**: Inter-annual variability in wind patterns (due to e.g. NAO oscillation) cannot be captured

## How to Improve This Project

1. **Add NWP wind speed**: use an OpenMeteo hourly wind speed forecast as an exogenous variable in Darts RegressionModel with uture_covariates
2. **Train on full 10-min data**: current pipeline resamples to hourly; sub-hourly Darts forecasts at 10-min may capture ramp events missed in hourly smoothing
3. **Physics-informed feature**: add ³ directly from measured wind speed; include the manufacturer power curve as a "physics baseline" predictor
4. **Curtailment masking**: identify curtailment periods (using the Theoretical_Power_Curve column) and exclude them from training
5. **Quantile regression**: produce P10/P90 wind intervals essential for balancing reserve calculations

## Production Considerations

1. **Model refresh cadence**: weekly retraining on rolling 60-day window to capture seasonal wind regime shifts
2. **NWP coupling**: consume GFS or ECMWF 6h-ahead wind forecasts at 100m hub height; adjust for local terrain correction factor
3. **Confidence intervals**: expose probabilistic forecasts (P10/P50/P90) for grid balancing and trading decisions
4. **Curtailment flag input**: when grid operator sends curtailment signal, override forecast with capped value
5. **Fleet generalisation**: train a global model across multiple turbines (multi-target or per-turbine) with transfer learning

## Common Mistakes to Avoid

1. **Using Theoretical_Power_Curve as a target**: this is the manufacturer's expected value, not the actual measured output — use it as a feature, not a label
2. **Ignoring curtailment periods**: including them trains the model to predict artificially low output during windy hours, degrading forecast quality
3. **Computing MAPE at zero-output (calm wind) observations**: uninformative and can dominate the metric; always use RMSE or MAE as primary metrics
4. **Using a fixed seasonal period**: if resampling, make sure SEASON constant is updated accordingly (96 at 15-min ≠ 24 at hourly)
5. **Not clipping negative predictions**: energy models should always clip to [0, rated_capacity] before reporting

## Mini Challenge / Exercises

1. **Add irradiance proxy**: read the Theoretical_Power_Curve column as a future covariate in a Darts LinearRegressionModel. Report RMSE change.
2. **Curtailment filter**: mask all rows where LV ActivePower (kW) < 0.3 × Theoretical_Power_Curve (KWh) and wind > 5 m/s. Retrain and report if RMSE improves.
3. **Longer horizon test**: increase HORIZON from 6 to 24. Plot how forecast skill (RMSE vs persistence) degrades with increasing look-ahead.
4. **Ramp event analysis**: define a "ramp" as ≥ 20% rated-power change in one hour. How many ramps are in the test set? What fraction are well-forecast?
5. **Winter vs summer**: split test set by season. Is the model more accurate in summer (stable) or winter (more wind)?

## Final Summary & Key Takeaways

### What We Did
- Downloaded 2 years of 10-min SCADA data from one wind turbine
- Resampled 10-min data to hourly; cleaned negatives and filled gaps
- Built wind-specific features including wind_cubed for the physical P ∝ v³ relationship
- Diagnosed curtailment events using the theoretical power curve column
- Benchmarked 40+ regressors with LazyPredict and ran FLAML AutoML (120s budget)
- Fitted Darts SeasonalNaive(24), ETS, and AutoARIMA models
- Identified top 3 by RMSE; analysed errors by wind speed bin

### Key Takeaways
1. **lag_24 (same hour yesterday) is the single most predictive feature** for wind — weather systems have day-scale persistence
2. **wind_cubed matters**: the P ∝ v³ relationship means wind speed is a strongly non-linear predictor that linear cores cannot represent with raw lag features alone
3. **Curtailment contaminates the training signal** — always detect and mask it before fitting
4. **MAPE is inappropriate** for wind at low-wind hours; use RMSE or MAE; for reporting use capacity-normalised RMSE (nRMSE = RMSE / rated_power)
5. **NWP is the silver bullet**: real operational wind forecasts use day-ahead NWP forecasts; pure historical models plateau quickly

---
*Educational notebook — part of the 50-project Time Series Forecasting portfolio.*